In [0]:
%pip install openpyxl xlrd tqdm -q

from pyspark.sql import functions as F
import pandas as pd
import io
import os
import time
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

In [0]:
%sql
USE CATALOG prd_mega;
USE SCHEMA scolom15;
SELECT current_catalog() AS catalog, current_schema() AS schema;

In [0]:
CONTROL_TABLE = "prd_mega.scolom15.file_classification_from2016to2019"
BRONZE_TABLE  = "prd_mega.scolom15.bronze_validaciones_from2016to2019"

In [0]:
# Load tables as pandas DataFrame for easier inspection  --> might be super heavy 
# classification_df = spark.table(CONTROL_TABLE).toPandas()
# bronze_df = spark.table(BRONZE_TABLE).toPandas()

In [0]:
# Inspect Columns of BRONZE_TABLE
spark.table(BRONZE_TABLE).columns
# Inspect Columns of CONTROL_TABLE
spark.table(CONTROL_TABLE).columns

In [0]:
# What values can classification_status take
spark.table(CONTROL_TABLE).select(F.col("classification_status")).distinct().show()
# What values can detection_notes take
spark.table(CONTROL_TABLE).select(F.col("detection_notes")).distinct().show()

In [0]:
# How many files sources? (count unique values of _source_file in BRONZE_TABLE)
in_bronze = spark.table(BRONZE_TABLE).select(F.col("_source_file")).distinct().count()
print(f"Number of unique files in BRONZE_TABLE: {in_bronze}")
# How many files sources? (count unique values of _source_file in CONTROL_TABLE)
in_control = spark.table(CONTROL_TABLE).select(F.col("raw_filepath")).distinct().count()
print(f"Number of unique files in CONTROL_TABLE: {in_control}")

# How many files sources with classification_status == "broken"? (count unique values of _source_file in CONTROL_TABLE)
in_control_broken = spark.table(CONTROL_TABLE).select(F.col("raw_filepath")).where(F.col("classification_status") == "broken").distinct().count()
print(f"Number of unique files in CONTROL_TABLE with classification_status == 'broken': {in_control_broken}")

# Check
assert in_bronze == in_control - in_control_broken

# Tabulate values of detection_notes
spark.table(CONTROL_TABLE).select(F.col("detection_notes")).groupBy(F.col("detection_notes")).count().show()

# Crosstabulate values of detection_notes and classification_status
spark.table(CONTROL_TABLE).groupBy(F.col("detection_notes"), F.col("classification_status")).count().show()

In [0]:
# Show sample of raw_filepath 
spark.table(CONTROL_TABLE).select(F.col("raw_filepath")).sample(False, 0.1).show(10)

# Assert there are no missings 
missing_rf = spark.table(CONTROL_TABLE).select(F.col("raw_filepath")).where(F.col("raw_filepath") == "").distinct().count()

assert missing_rf == 0
print(f"Number of missing raw_filepath: {missing_rf}")

In [0]:
# check that each variable has the right format 
for col in spark.table(BRONZE_TABLE).columns:
  print(col, spark.table(BRONZE_TABLE).select(F.col(col)).dtypes)


In [0]:
# Show random sample of fecha_transaccion  
display(spark.table(BRONZE_TABLE).select(F.col("fecha_transaccion")).sample(False, 0.1))

# Show last 100 observations for fecha_transaccion
display(
    spark.table(BRONZE_TABLE)
    .select(F.col("fecha_transaccion"))
    .orderBy(F.col("fecha_transaccion").desc())
    .limit(100)
)

In [0]:
# Show random sample of fecha_transaccion  
display(spark.table(BRONZE_TABLE).select(F.col("station_access")).sample(False, 0.1))

# Show last 100 observations for station_access
display(
    spark.table(BRONZE_TABLE)
    .select(F.col("station_access"))
    .orderBy(F.col("station_access").desc())
    .limit(100)

In [0]:
# Tag observations that contain weird caracters (outside letters and numbers)
def contains_non_normal_characters(text):
    import re
    pattern = r'[^A-Za-zÑñ0-9_\-/: ]'
    return False if text is None else bool(re.search(pattern, text))

# UDF for Spark
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

contains_non_normal_characters_udf = udf(contains_non_normal_characters, BooleanType())

# Filter rows in BRONZE_TABLE where fecha_transaccion contains non-normal characters and list _source_file
weird_rows = (
    spark.table(BRONZE_TABLE)
    .filter(contains_non_normal_characters_udf(F.col("fecha_transaccion")))
    .select(F.col("_source_file"), F.col("fecha_transaccion"))
    .distinct()
)

display(weird_rows)

In [0]:
# Tag observations that contain letters (OTHER THAN UTC) in the variable fecha_transaccion
def contains_letters_except_utc(text):
    import re
    if text is None:
        return False
    # Remove 'UTC' (case-insensitive) from text
    cleaned = re.sub(r'(?i)UTC', '', text)
    # Check for any remaining letters
    return bool(re.search(r'[A-Za-z]', cleaned))

from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

contains_letters_except_utc_udf = udf(contains_letters_except_utc, BooleanType())

tagged_rows = (
    spark.table(BRONZE_TABLE)
    .filter(contains_letters_except_utc_udf(F.col("fecha_transaccion")))
    .select(F.col("_source_file"), F.col("fecha_transaccion"))
    .distinct()
)

display(tagged_rows)

# it's only 12 rows
